# Leaderboard — Top 5 (BTC 1D Transformer OHLCV)

Este notebook lee automáticamente las métricas guardadas en `experimentos/modelos/*/metrics_transformer_btc_1d.json` y construye un **Top 5**.

Además:
- calcula **mejora (%)** contra baselines simples,
- estima un **IC 95%** (Wilson) para `logret_dir` usando el tamaño del test.

Se muestran dos rankings:

**A) Ranking por error (winner_error)**
1) menor `logret_rmse`,
2) menor `logret_mae`,
3) mayor `logret_dir`.

**B) Ranking por direccionalidad (winner_dir)**
1) mayor `logret_dir`,
2) menor `logret_rmse`,
3) menor `logret_mae`.


In [6]:
import json
import pathlib
import numpy as np
import pandas as pd


In [7]:
def find_repo_root():
    start = pathlib.Path(".").resolve()
    for p in [start] + list(start.parents):
        if (p / "experimentos" / "modelos").exists():
            return p
        if (p / "btc-forecast" / "experimentos" / "modelos").exists():
            return p / "btc-forecast"
    raise FileNotFoundError(f"No se encontró el repo raíz desde: {start}")

repo_root = find_repo_root()
models_root = repo_root / "experimentos" / "modelos"
models_root


PosixPath('/Users/williamvasquez/Library/CloudStorage/OneDrive-Personal/Documentos/William/cursos Online/Masters/IA VIU/trabajo fin master/proyecto_grado/experimentos/modelos')

In [8]:
def load_run_metrics(metrics_path: pathlib.Path):
    with metrics_path.open() as f:
        data = json.load(f)
    test = data.get("test", {})
    m_log = test.get("test_model_logret", {})
    m_ohl = test.get("test_model_ohlcv", {})
    b_log0 = test.get("test_logret_zero", {})
    b_logp = test.get("test_logret_persistence", {})
    b_ohlp = test.get("test_ohlcv_persistence", {})
    return {
        "run_tag": metrics_path.parent.name,
        "logret_mae": m_log.get("mae"),
        "logret_rmse": m_log.get("rmse"),
        "logret_dir": m_log.get("directional_acc"),
        "close_rmse": m_ohl.get("rmse_close"),
        "volume_rmse": m_ohl.get("rmse_volume"),
        "baseline_logret_rmse_zero": b_log0.get("rmse"),
        "baseline_logret_rmse_persist": b_logp.get("rmse"),
        "baseline_close_rmse_persist": b_ohlp.get("rmse_close"),
        "metrics_path": str(metrics_path),
    }


In [9]:
rows = []
for metrics_path in models_root.glob("*/metrics_transformer_btc_1d.json"):
    try:
        rows.append(load_run_metrics(metrics_path))
    except Exception as e:
        rows.append({"run_tag": metrics_path.parent.name, "error": str(e), "metrics_path": str(metrics_path)})

df = pd.DataFrame(rows)
for c in [
    "logret_mae","logret_rmse","logret_dir","close_rmse","volume_rmse",
    "baseline_logret_rmse_zero","baseline_logret_rmse_persist","baseline_close_rmse_persist",
]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

def safe_improvement_pct(baseline, model):
    if baseline is None or model is None:
        return np.nan
    baseline = float(baseline)
    model = float(model)
    if not np.isfinite(baseline) or not np.isfinite(model) or baseline == 0.0:
        return np.nan
    return 100.0 * (baseline - model) / baseline

preds_root = repo_root / "experimentos" / "predicciones"
n_test = []
for rt in df["run_tag"].astype(str).tolist():
    p = preds_root / rt / "btc_1d_transformer_test_predictions.csv"
    if p.exists():
        try:
            n_test.append(int(pd.read_csv(p, usecols=[0]).shape[0]))
        except Exception:
            n_test.append(np.nan)
    else:
        n_test.append(np.nan)
df["n_test"] = n_test

def wilson_ci(p, n, z=1.96):
    if p is None or n is None:
        return (np.nan, np.nan)
    p = float(p)
    n = float(n)
    if not np.isfinite(p) or not np.isfinite(n) or n <= 0:
        return (np.nan, np.nan)
    denom = 1.0 + (z**2) / n
    center = (p + (z**2) / (2.0 * n)) / denom
    half = (z * np.sqrt((p * (1.0 - p) / n) + (z**2) / (4.0 * n**2))) / denom
    return (max(0.0, center - half), min(1.0, center + half))

df["impr_logret_rmse_vs_zero_pct"] = df.apply(lambda r: safe_improvement_pct(r.get("baseline_logret_rmse_zero"), r.get("logret_rmse")), axis=1)
df["impr_close_rmse_vs_persist_pct"] = df.apply(lambda r: safe_improvement_pct(r.get("baseline_close_rmse_persist"), r.get("close_rmse")), axis=1)
df[["dir_ci95_low","dir_ci95_high"]] = df.apply(lambda r: pd.Series(wilson_ci(r.get("logret_dir"), r.get("n_test"))), axis=1)

df_error = df.sort_values(["logret_rmse", "logret_mae", "logret_dir"], ascending=[True, True, False], na_position="last")
df_error = df_error.reset_index(drop=True)
df_error["rank"] = np.arange(1, len(df_error) + 1)
df_error["winner_error"] = df_error["rank"] == 1
top5_error = df_error.head(5).copy()

df_dir = df.sort_values(["logret_dir", "logret_rmse", "logret_mae"], ascending=[False, True, True], na_position="last")
df_dir = df_dir.reset_index(drop=True)
df_dir["rank_dir"] = np.arange(1, len(df_dir) + 1)
df_dir["winner_dir"] = df_dir["rank_dir"] == 1
top5_dir = df_dir.head(5).copy()

from IPython.display import display
cols = [
    "run_tag","logret_rmse","logret_mae","logret_dir","dir_ci95_low","dir_ci95_high","n_test",
    "impr_logret_rmse_vs_zero_pct","close_rmse","impr_close_rmse_vs_persist_pct",
    "rank","winner_error",
]
display(top5_error[cols].copy())

cols_dir = [
    "run_tag","logret_dir","dir_ci95_low","dir_ci95_high","n_test",
    "logret_rmse","logret_mae","impr_logret_rmse_vs_zero_pct",
    "rank_dir","winner_dir",
]
display(top5_dir[cols_dir].copy())
top5_dir[cols_dir].copy()


,run_tag,logret_rmse,logret_mae,logret_dir,dir_ci95_low,dir_ci95_high,n_test,impr_logret_rmse_vs_zero_pct,close_rmse,impr_close_rmse_vs_persist_pct,rank,winner_error
0,exp3_epochs300,0.023562,0.016387,0.511468,0.464652,0.558084,436,0.225532,2183.091553,0.098026,1,True
1,exp1_window14,0.023583,0.016351,0.489655,0.442974,0.536517,435,0.228633,2185.769043,0.060123,2,False
2,exp9_exp6_minEpoch60,0.023644,0.016320,0.527523,0.480625,0.573940,436,-0.121371,2186.656006,-0.065090,3,False
3,exp5_dmodel128_heads8,0.023659,0.016341,0.500000,0.453272,0.546728,436,-0.185448,2192.947266,-0.352988,4,False
4,exp2_deeper4,0.023679,0.016476,0.506881,0.460097,0.553544,436,-0.267469,2192.098389,-0.314142,5,False


,run_tag,logret_dir,dir_ci95_low,dir_ci95_high,n_test,logret_rmse,logret_mae,impr_logret_rmse_vs_zero_pct,rank_dir,winner_dir
0,exp9_exp6_minEpoch60,0.527523,0.480625,0.573940,436,0.023644,0.016320,-0.121371,1,True
1,exp3_epochs300,0.511468,0.464652,0.558084,436,0.023562,0.016387,0.225532,2,False
2,exp8_exp4_minEpoch60,0.510345,0.463483,0.557026,435,0.023784,0.016440,-0.625204,3,False
3,exp7_window30,0.508083,0.461130,0.554894,433,0.023767,0.016644,-0.360331,4,False
4,exp4_window14_deeper4,0.508046,0.461200,0.554751,435,0.024125,0.017196,-2.065141,5,False


,run_tag,logret_dir,dir_ci95_low,dir_ci95_high,n_test,logret_rmse,logret_mae,impr_logret_rmse_vs_zero_pct,rank_dir,winner_dir
0,exp9_exp6_minEpoch60,0.527523,0.480625,0.573940,436,0.023644,0.016320,-0.121371,1,True
1,exp3_epochs300,0.511468,0.464652,0.558084,436,0.023562,0.016387,0.225532,2,False
2,exp8_exp4_minEpoch60,0.510345,0.463483,0.557026,435,0.023784,0.016440,-0.625204,3,False
3,exp7_window30,0.508083,0.461130,0.554894,433,0.023767,0.016644,-0.360331,4,False
4,exp4_window14_deeper4,0.508046,0.461200,0.554751,435,0.024125,0.017196,-2.065141,5,False


In [10]:
def load_metrics_full(run_tag: str):
    p = models_root / run_tag / "metrics_transformer_btc_1d.json"
    with p.open() as f:
        data = json.load(f)
    test = data.get("test", {})
    m_ohl = test.get("test_model_ohlcv", {})
    b_p_ohl = test.get("test_ohlcv_persistence", {})
    b_l_ohl = test.get("test_ohlcv_linear", {})
    m_log = test.get("test_model_logret", {})
    b_p_log = test.get("test_logret_persistence", {})
    b_l_log = test.get("test_logret_linear", {})
    return {
        "run_tag": run_tag,
        "rmse_mean_model": m_ohl.get("rmse_mean"),
        "rmse_mean_persist": b_p_ohl.get("rmse_mean"),
        "rmse_mean_linear": b_l_ohl.get("rmse_mean"),
        "rmse_close_model": m_ohl.get("rmse_close"),
        "rmse_close_persist": b_p_ohl.get("rmse_close"),
        "rmse_close_linear": b_l_ohl.get("rmse_close"),
        "rmse_volume_model": m_ohl.get("rmse_volume"),
        "rmse_volume_persist": b_p_ohl.get("rmse_volume"),
        "rmse_volume_linear": b_l_ohl.get("rmse_volume"),
        "logret_rmse_model": m_log.get("rmse"),
        "logret_dir_model": m_log.get("directional_acc"),
        "logret_rmse_persist": b_p_log.get("rmse"),
        "logret_rmse_linear": b_l_log.get("rmse"),
    }

def improvement_pct(baseline, model):
    if baseline is None or model is None:
        return np.nan
    baseline = float(baseline)
    model = float(model)
    if not np.isfinite(baseline) or baseline == 0 or not np.isfinite(model):
        return np.nan
    return 100.0 * (baseline - model) / baseline

runs_error = top5_error["run_tag"].astype(str).tolist()
full = pd.DataFrame([load_metrics_full(rt) for rt in runs_error])

full["impr_rmse_mean_vs_persist_pct"] = full.apply(lambda r: improvement_pct(r["rmse_mean_persist"], r["rmse_mean_model"]), axis=1)
full["impr_rmse_close_vs_persist_pct"] = full.apply(lambda r: improvement_pct(r["rmse_close_persist"], r["rmse_close_model"]), axis=1)
full["impr_rmse_volume_vs_persist_pct"] = full.apply(lambda r: improvement_pct(r["rmse_volume_persist"], r["rmse_volume_model"]), axis=1)

tabla_ohlcv = full[[
    "run_tag",
    "rmse_mean_model","rmse_mean_persist","rmse_mean_linear","impr_rmse_mean_vs_persist_pct",
    "rmse_close_model","rmse_close_persist","rmse_close_linear","impr_rmse_close_vs_persist_pct",
    "rmse_volume_model","rmse_volume_persist","rmse_volume_linear","impr_rmse_volume_vs_persist_pct",
]].copy()

tabla_ohlcv = tabla_ohlcv.round({
    "rmse_mean_model": 3,
    "rmse_mean_persist": 3,
    "rmse_mean_linear": 3,
    "impr_rmse_mean_vs_persist_pct": 2,
    "rmse_close_model": 3,
    "rmse_close_persist": 3,
    "rmse_close_linear": 3,
    "impr_rmse_close_vs_persist_pct": 2,
    "rmse_volume_model": 3,
    "rmse_volume_persist": 3,
    "rmse_volume_linear": 3,
    "impr_rmse_volume_vs_persist_pct": 2,
})

display(tabla_ohlcv)
def df_to_markdown_pipe(df: pd.DataFrame) -> str:
    cols = [str(c) for c in df.columns]
    is_num = {c: pd.api.types.is_numeric_dtype(df[c]) for c in df.columns}

    def esc(v):
        if v is None or (isinstance(v, float) and np.isnan(v)):
            return ""
        s = str(v)
        return s.replace("|", "\\|")

    header = "| " + " | ".join(cols) + " |"
    align = "| " + " | ".join(["---:" if is_num.get(c, False) else "---" for c in df.columns]) + " |"
    rows = []
    for _, r in df.iterrows():
        rows.append("| " + " | ".join([esc(r[c]) for c in df.columns]) + " |")
    return "\n".join([header, align] + rows)

print(df_to_markdown_pipe(tabla_ohlcv))


,run_tag,rmse_mean_model,rmse_mean_persist,rmse_mean_linear,impr_rmse_mean_vs_persist_pct,rmse_close_model,rmse_close_persist,rmse_close_linear,impr_rmse_close_vs_persist_pct,rmse_volume_model,rmse_volume_persist,rmse_volume_linear,impr_rmse_volume_vs_persist_pct
0,exp3_epochs300,3354.938,4470.879,4995.134,24.96,2183.092,2185.234,2377.863,0.10,11354.038,13863.955,19252.080,18.10
1,exp1_window14,3420.297,4471.114,4990.094,23.50,2185.769,2187.084,2472.209,0.06,11647.545,13873.867,19090.504,16.05
2,exp9_exp6_minEpoch60,3525.842,4470.879,4995.134,21.14,2186.656,2185.234,2377.863,-0.07,12082.645,13863.955,19252.080,12.85
3,exp5_dmodel128_heads8,3429.512,4470.879,4995.134,23.29,2192.947,2185.234,2377.863,-0.35,11676.679,13863.955,19252.080,15.78
4,exp2_deeper4,3382.048,4470.879,4995.134,24.35,2192.098,2185.234,2377.863,-0.31,11400.301,13863.955,19252.080,17.77


| run_tag | rmse_mean_model | rmse_mean_persist | rmse_mean_linear | impr_rmse_mean_vs_persist_pct | rmse_close_model | rmse_close_persist | rmse_close_linear | impr_rmse_close_vs_persist_pct | rmse_volume_model | rmse_volume_persist | rmse_volume_linear | impr_rmse_volume_vs_persist_pct |
| --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| exp3_epochs300 | 3354.938 | 4470.879 | 4995.134 | 24.96 | 2183.092 | 2185.234 | 2377.863 | 0.1 | 11354.038 | 13863.955 | 19252.08 | 18.1 |
| exp1_window14 | 3420.297 | 4471.114 | 4990.094 | 23.5 | 2185.769 | 2187.084 | 2472.209 | 0.06 | 11647.545 | 13873.867 | 19090.504 | 16.05 |
| exp9_exp6_minEpoch60 | 3525.842 | 4470.879 | 4995.134 | 21.14 | 2186.656 | 2185.234 | 2377.863 | -0.07 | 12082.645 | 13863.955 | 19252.08 | 12.85 |
| exp5_dmodel128_heads8 | 3429.512 | 4470.879 | 4995.134 | 23.29 | 2192.947 | 2185.234 | 2377.863 | -0.35 | 11676.679 | 13863.955 | 19252.08 | 15.78 |
| exp2_deeper4 | 3382.048 | 4470.

## Enlaces rápidos a artefactos

Por cada `run_tag`, revisa:
- `experimentos/modelos/<run_tag>/config_transformer_btc_1d.json`
- `experimentos/modelos/<run_tag>/metrics_transformer_btc_1d.json`
- `experimentos/figuras/<run_tag>/` (imágenes)
- `experimentos/predicciones/<run_tag>/btc_1d_transformer_test_predictions.csv`
